# Chunking Strategy Ablation Study

Compares chunking configurations and embedding models using RAGAS metrics, tracked with MLflow.

**What we vary:**
- `chunk_size`: 300, 500, 800
- `chunk_overlap`: 50, 75, 100
- `embed_model`: MiniLM-L6 vs BGE-small

**Metrics logged per run:**
- `faithfulness`, `answer_relevancy`, `context_precision` (RAGAS)
- `avg_chunk_size`, `num_chunks`, `avg_retrieval_score`

In [ ]:
import os, sys
sys.path.insert(0, '../app')

import numpy as np
import faiss
import mlflow
from itertools import product
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from langchain_groq import ChatGroq
from groq import Groq
from rag_engine import _fetch_pubmed_abstracts, generate as groq_generate

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
groq_client  = Groq(api_key=GROQ_API_KEY)

In [ ]:
# ── Fetch raw docs once (reused across all runs) ──────────────
print('Fetching PubMed abstracts...')
raw_docs = _fetch_pubmed_abstracts(
    'guselkumab Tremfya psoriasis clinical trial', max_results=60
)
print(f'Fetched {len(raw_docs)} abstracts')

# Eval questions
EVAL_QUESTIONS = [
    'What is the mechanism of action of guselkumab?',
    'What are the most common adverse effects of Tremfya?',
    'How effective is guselkumab for psoriatic arthritis?',
    'What is the recommended dosage for guselkumab?',
    'What cytokine does guselkumab selectively inhibit?',
]

In [ ]:
def build_index(raw_docs, chunk_size, chunk_overlap, embed_model_name):
    """Build FAISS index for a given config."""
    # Chunk
    lc_docs = [
        Document(
            page_content=f"Title: {d['title']}\n\nAbstract: {d['abstract']}",
            metadata={'pmid': d['pmid'], 'title': d['title'], 'year': d['year']}
        ) for d in raw_docs
    ]
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    chunks = splitter.split_documents(lc_docs)

    # Embed
    model = SentenceTransformer(embed_model_name)
    texts = [c.page_content for c in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True).astype(np.float32)

    # Index
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index, chunks, model, texts


def retrieve_with_index(query, index, chunks, model, k=4):
    q_vec = model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_vec, k)
    return [
        {'score': float(scores[0][i]),
         'content': chunks[indices[0][i]].page_content,
         'metadata': chunks[indices[0][i]].metadata}
        for i in range(k)
    ]


def build_eval_dataset(questions, index, chunks, embed_model):
    rows = {'question': [], 'answer': [], 'contexts': [], 'reference': []}
    for q in questions:
        ctx = retrieve_with_index(q, index, chunks, embed_model)
        ans = groq_generate(q, ctx)
        rows['question'].append(q)
        rows['answer'].append(ans)
        rows['contexts'].append([c['content'] for c in ctx])
        rows['reference'].append('')
    return Dataset.from_dict(rows)

In [ ]:
# ── Experiment grid ───────────────────────────────────────────
CHUNK_SIZES    = [300, 500, 800]
CHUNK_OVERLAPS = [50, 75]
EMBED_MODELS   = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'BAAI/bge-small-en-v1.5',
]

judge_llm = LangchainLLMWrapper(
    ChatGroq(model='llama-3.3-70b-versatile', api_key=GROQ_API_KEY, temperature=0)
)

mlflow.set_experiment('pharma-rag-chunking-ablation')

all_results = []

for chunk_size, chunk_overlap, embed_model_name in product(CHUNK_SIZES, CHUNK_OVERLAPS, EMBED_MODELS):
    run_name = f'cs{chunk_size}_co{chunk_overlap}_{embed_model_name.split("/")[-1]}'
    print(f'\n🔬 Run: {run_name}')

    with mlflow.start_run(run_name=run_name):
        # Log hyperparams
        mlflow.log_params({
            'chunk_size':    chunk_size,
            'chunk_overlap': chunk_overlap,
            'embed_model':   embed_model_name,
            'retrieval_k':   4,
            'llm':           'llama-3.3-70b-versatile',
        })

        # Build index
        index, chunks, model, texts = build_index(
            raw_docs, chunk_size, chunk_overlap, embed_model_name
        )

        # Log index stats
        mlflow.log_metrics({
            'num_chunks':     len(chunks),
            'avg_chunk_size': int(np.mean([len(t) for t in texts])),
        })

        # Build eval dataset
        dataset = build_eval_dataset(EVAL_QUESTIONS, index, chunks, model)

        # RAGAS evaluation
        results = evaluate(
            dataset=dataset,
            metrics=[faithfulness, answer_relevancy, context_precision],
            llm=judge_llm,
        )
        scores = results.to_pandas()[[
            'faithfulness', 'answer_relevancy', 'context_precision'
        ]].mean().to_dict()

        # Log RAGAS scores
        mlflow.log_metrics(scores)

        print(f'   faithfulness={scores["faithfulness"]:.3f}  '
              f'relevancy={scores["answer_relevancy"]:.3f}  '
              f'precision={scores["context_precision"]:.3f}')

        all_results.append({'run': run_name, **scores})

print('\n✅ All runs complete. Open MLflow UI: mlflow ui')

In [ ]:
# ── Results summary ───────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_results).sort_values('faithfulness', ascending=False)
print('\n📊 Results ranked by faithfulness:')
print(df.to_string(index=False))